# Module 1 — Agentic RAG — Homework

Building a RAG system from scratch over the course **lessons**, then making it agentic.

I used **Claude (Anthropic, `claude-haiku-4-5`)** as the LLM instead of OpenAI — the homework allows any provider, so I adapted the client and the usage fields accordingly. Numbers for the token/agent questions are therefore "closest option".

## Setup

```bash
uv add gitsource minsearch anthropic
```

In [1]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
import anthropic

MODEL = "claude-haiku-4-5"
QUERY = "How does the agentic loop keep calling the model until it stops?"

## Preparation — pull the lesson pages

We pull every lesson markdown file straight from the course repo, pinned to commit `8c1834d` so everyone works with identical data. The `filename_filter` keeps only files under a module's `lessons/` folder.

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []
for file in files:
    documents.append(file.parse())

documents[0]["filename"], documents[0]["content"][:120]

('01-agentic-rag/lessons/01-intro.md',
 '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8')

## Q1. How many lesson pages

Each `parse()` returns a dict with `filename` and `content`, one per lesson page.

In [3]:
len(documents)

72

**Answer Q1: 72**

## Q2. Indexing and searching

Index with minsearch: `content` as a text field (tokenized + ranked), `filename` as a keyword field (exact). Then search.

In [4]:
index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

results = index.search(QUERY, num_results=5)
results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

**Answer Q2: `01-agentic-rag/lessons/14-agentic-loop.md`**

## Q3. RAG

RAG flow: **search → build context → build prompt → LLM**. `RAGBase` from the lessons was written for the FAQ schema (`section`/`question`/`answer`); our docs are `filename`/`content`, so I reimplement `search` and `build_context`.

To read token usage I return the whole response and pull `response.usage.input_tokens` (the Anthropic field; OpenAI calls it `prompt_tokens`).

In [5]:
INSTRUCTIONS = (
    "You are a course teaching assistant. Answer the student's question using "
    "only the provided context. If the answer is not in the context, say "
    '"I don\'t know."'
)

PROMPT_TEMPLATE = "QUESTION: {question}\n\nCONTEXT:\n{context}"


class RAG:
    def __init__(self, index, client, model=MODEL):
        self.index = index
        self.client = client
        self.model = model

    def search(self, query):
        return self.index.search(query, num_results=5)

    def build_context(self, results):
        return "\n\n".join(
            f"# {d['filename']}\n{d['content']}" for d in results
        ).strip()

    def llm(self, prompt):
        return self.client.messages.create(
            model=self.model,
            max_tokens=1024,
            system=INSTRUCTIONS,
            messages=[{"role": "user", "content": prompt}],
        )

    def rag(self, query):
        results = self.search(query)
        prompt = PROMPT_TEMPLATE.format(
            question=query, context=self.build_context(results)
        )
        response = self.llm(prompt)
        answer = next(b.text for b in response.content if b.type == "text")
        return answer, response.usage.input_tokens


client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment

rag = RAG(index, client)
answer, tokens_q3 = rag.rag(QUERY)
print("input tokens:", tokens_q3)
print(answer)

input tokens: 8176
# How the Agentic Loop Keeps Calling the Model Until It Stops

Based on the context, the agentic loop uses a **`while True` loop with a simple exit condition** to keep calling the model until it stops.

Here's how it works:

## The Core Mechanism

The loop maintains a `has_function_calls` flag that tracks whether the model's response contains any function calls:

```python
while True:
    # Call the model
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )
    
    messages.extend(response.output)
    has_function_calls = False
    
    # Process the response
    for item in response.output:
        if item.type == "function_call":
            # Execute the function and add result to messages
            has_function_calls = True
        elif item.type == "message":
            # Store the final answer
            last_answer = item.content[0].text
    
    # Exit condition: break if 

**Answer Q3: ~7000** (whole pages stuffed into context make a large prompt).

## Q4. Chunking

Long pages hurt retrieval precision. A sliding window of `size=2000` chars moving in `step=1000` splits each page into overlapping chunks (overlap = `size - step` = 1000 chars, so passages crossing a boundary still appear whole somewhere).

In [6]:
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

**Answer Q4: 295**

## Q5. RAG with chunking

Index the chunks the same way and point the RAG at them. Same query, same way of reading input tokens — but the context is now small chunks instead of whole pages.

In [7]:
chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

rag_chunked = RAG(chunk_index, client)
answer, tokens_q5 = rag_chunked.rag(QUERY)
print("input tokens:", tokens_q5)
print("reduction vs Q3:", round(tokens_q3 / tokens_q5, 1), "x fewer")

input tokens: 2632
reduction vs Q3: 3.1 x fewer


**Answer Q5: ~3× fewer** (the chunked context is roughly a third of the whole-page context).

## Q6. Turning it into an agent

So far `search` runs once with the exact query. Now we give the LLM a `search` **tool** and let it decide when and what to search. The agentic loop keeps calling the model, executing any tool calls and feeding the results back, until the model stops asking for tools (`stop_reason != "tool_use"`).

We wrap `search` in a counter to see how many times the agent calls it.

In [8]:
search_calls = 0


def search(query: str) -> str:
    """Search the course lessons for passages relevant to the query."""
    global search_calls
    search_calls += 1
    results = chunk_index.search(query, num_results=5)
    return "\n\n".join(f"# {d['filename']}\n{d['content']}" for d in results)


tools = [{
    "name": "search",
    "description": "Search the course lessons knowledge base and return the most relevant chunks.",
    "input_schema": {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "search keywords"}},
        "required": ["query"],
    },
}]

agent_instructions = (
    "You're a course teaching assistant. Answer the student's question using the "
    "search tool. Make multiple searches with different keywords before answering."
)

question = "How does the agentic loop work, and how is it different from plain RAG?"
messages = [{"role": "user", "content": question}]

while True:
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=agent_instructions,
        tools=tools,
        messages=messages,
    )
    if response.stop_reason != "tool_use":
        break
    messages.append({"role": "assistant", "content": response.content})
    tool_results = []
    for block in response.content:
        if block.type == "tool_use" and block.name == "search":
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": search(block.input["query"]),
            })
    messages.append({"role": "user", "content": tool_results})

final = next(b.text for b in response.content if b.type == "text")
print("search tool calls:", search_calls)
print(final)

search tool calls: 3
Great! I found comprehensive information about both topics. Let me explain the agentic loop and how it differs from plain RAG.

## How the Agentic Loop Works

The **agentic loop** is a `while` loop that continuously calls an LLM (Large Language Model) until it produces a final answer. Here's how it works:

### The Three Components of an Agent

1. **Instructions** - A developer prompt that defines the agent's role and desired behavior
2. **Tools** - Functions the agent can call (like a search function)
3. **Memory** - Message history that tracks everything the agent has tried

### The Loop Process

```python
while True:
    # 1. Call the LLM
    response = llm(messages)
    messages.extend(response)
    
    # 2. Check if there are function calls to make
    for item in response:
        if item.type == "function_call":
            # Execute the tool
            result = execute_tool(item)
            messages.append(result)
            has_function_calls = True
   

**Answer Q6: ~4** (the agent decides on its own to search a few times with different keywords, vs plain RAG's single fixed search).

## Summary of answers

| Question | Answer |
|---|---|
| Q1 | 72 |
| Q2 | `01-agentic-rag/lessons/14-agentic-loop.md` |
| Q3 | 7000 |
| Q4 | 295 |
| Q5 | 3× fewer |
| Q6 | 4 |